# Do Earthworms Make Sounds? 🪱🔊
### Decoding the Vibrations of Soil Life with Python, PyCon Tanzania 2026

**Samuel Babalola**, Ruzivo Research Lab, African Leadership University

This notebook runs live during the talk. It covers the part of the pipeline that usually gets skipped: **turning unlabelled soil recordings into a training set we can trust.**

| Step | What happens | Python tools |
|---|---|---|
| 1 | Listen to busy vs quiet soil, and see their mel spectrograms | `librosa`, `IPython.display` |
| 2 | Turn sound into numbers (ecoacoustic indices) | `librosa`, `numpy`, `scipy` |
| 3 | **Labeller 1:** a percentile threshold | `pandas` |
| 4 | **Labeller 2:** let the data cluster itself | `scikit-learn` |
| 5 | Do the two labellers agree? | `scikit-learn` |
| 6 | Settle the disagreements | `scikit-learn` |
| 7 | *(optional)* Ask the deployed model | `requests` |

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from IPython.display import Audio, display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Paths (relative to this notebook)
AUDIO_DIR    = Path("../sample_clips")                 # a handful of .wav clips for the live demo
FEATURES_CSV = Path("../data/features_extended.csv")   # features for the full dataset (588 clips)
API_URL      = "https://sammyg123-faunapulse.hf.space" # deployed model (Hugging Face Space)

HIGH_CLIP = AUDIO_DIR / "B363T8_Clean_Left_221206-101604-420_480.wav"   # a very active clip
LOW_CLIP  = AUDIO_DIR / "B363T3_Clean_Right_221130-095528-960_1020.wav"  # a very quiet clip

FEATURES = ["bioacoustic_index", "event_density"]     # the two indices used for labelling
SEED = 42

# Big, readable plots for screen sharing
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 14, "axes.titlesize": 16})
sns.set_theme(style="whitegrid", font_scale=1.2)

## 1. Listen to the soil

Each clip is a recording from a **piezoelectric disc** pressed into the soil. It picks up vibrations, not airborne sound.

In [ ]:
clips = sorted(p for p in AUDIO_DIR.glob("*") if p.suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"})
print(f"{len(clips)} clips found in {AUDIO_DIR}\n")

y_high, sr = librosa.load(HIGH_CLIP, sr=None)
y_low, _   = librosa.load(LOW_CLIP, sr=None)
print(f"Each clip: {len(y_high) / sr:.0f} s at {sr} Hz")

🔊 **A busy patch of soil**

In [ ]:
display(Audio(y_high, rate=sr))

🔇 **A quiet patch of soil**

In [ ]:
display(Audio(y_low, rate=sr))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
librosa.display.waveshow(y_high, sr=sr, ax=axes[0], color="#d1495b")
axes[0].set_title("Busy soil")
librosa.display.waveshow(y_low, sr=sr, ax=axes[1], color="#30638e")
axes[1].set_title("Quiet soil")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

### What the model sees: mel spectrograms

A **mel spectrogram** shows which frequencies are active over time. These images are what the classifier (InceptionV3) was trained on. The settings below match the deployed API.

In [ ]:
def mel_db(y, sr):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=320, n_mels=64)
    return librosa.power_to_db(S, ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (title, y_) in zip(axes, [("Busy soil", y_high), ("Quiet soil", y_low)]):
    img = librosa.display.specshow(mel_db(y_, sr), sr=sr, hop_length=320,
                                   x_axis="time", y_axis="mel", ax=ax, cmap="magma")
    ax.set_title(title)
fig.colorbar(img, ax=axes, format="%+2.0f dB")
plt.show()

We can hear and see the difference in these two, but nobody has labelled the other **586 clips**.
**How do we turn hundreds of recordings like these into training data?**

## 2. Turn sound into numbers

Ecologists already have tools for this: **ecoacoustic indices**. We compute two main ones:

- **Bioacoustic index:** energy in the frequency band where biological activity tends to sit
- **Event density:** how many distinct "events" (onsets) happen per second

In [ ]:
def bioacoustic_index(S, sr, fmin=2000, fmax=8000):
    """Total spectral energy in the biologically relevant band."""
    freqs = librosa.fft_frequencies(sr=sr)
    band = (freqs >= fmin) & (freqs <= fmax)
    return S[band, :].sum()


def event_density(y, sr, hop_length=512):
    """Number of acoustic events (onsets) per second."""
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop_length)
    return len(onsets) / (len(y) / sr)


def spectral_entropy(S):
    """How evenly energy is spread across frequencies (mean over frames)."""
    p = S / (S.sum(axis=0, keepdims=True) + 1e-12)
    return entropy(p, base=2, axis=0).mean()


def extract_features(path):
    y, sr = librosa.load(path, sr=None)
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
    return {
        "filename": path.name,
        "rms": librosa.feature.rms(y=y).mean(),
        "spectral_entropy": spectral_entropy(S),
        "bioacoustic_index": bioacoustic_index(S, sr),
        "event_density": event_density(y, sr),
    }

In [ ]:
sample_features = pd.DataFrame([extract_features(p) for p in clips])
sample_features.round(3)

## 3. Labeller 1: a percentile threshold (`pandas`)

Now we scale up to the **full dataset**. The same `extract_features` function was run on every clip ahead of time.

1. Min-max normalise both indices
2. Average them into one **activity score**
3. Call the top 25% **High** activity

In [ ]:
if FEATURES_CSV.exists():
    df = pd.read_csv(FEATURES_CSV)
    print(f"Full dataset: {len(df)} clips")
else:
    df = sample_features.copy()
    print(f"{FEATURES_CSV} not found, so using the {len(df)} sample clips")

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

df["activity_score"] = (minmax(df["bioacoustic_index"]) + minmax(df["event_density"])) / 2

threshold = df["activity_score"].quantile(0.75)
df["threshold_label"] = np.where(df["activity_score"] >= threshold, "High", "Low")

sns.histplot(df["activity_score"], bins=40, kde=True, color="skyblue")
plt.axvline(threshold, color="red", linestyle="--", linewidth=2, label=f"75th percentile = {threshold:.3f}")
plt.title("Activity score across all clips")
plt.xlabel("Activity score")
plt.legend()
plt.show()

df["threshold_label"].value_counts()

Simple and transparent, but **arbitrary**. Why the 75th percentile and not the 70th?
We need a second opinion that doesn't depend on that choice.

## 4. Labeller 2: let the data cluster itself (`scikit-learn`)

Same two features, no threshold: standardise, project with PCA, and ask **KMeans** for two groups.
Then name the clusters by which one has the higher average activity score.

In [ ]:
X_std = StandardScaler().fit_transform(df[FEATURES])
X_pca = PCA(n_components=2, random_state=SEED).fit_transform(X_std)

kmeans = KMeans(n_clusters=2, n_init="auto", random_state=SEED).fit(X_pca)
df["cluster"] = kmeans.labels_

high_cluster = df.groupby("cluster")["activity_score"].mean().idxmax()
df["cluster_label"] = np.where(df["cluster"] == high_cluster, "High", "Low")

sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df["cluster_label"],
                palette={"High": "#d1495b", "Low": "#30638e"}, s=60, alpha=0.8)
plt.title("KMeans clusters (k = 2) in PCA space")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

df["cluster_label"].value_counts()

**Why two clusters?** A quick check with the silhouette score *(optional, skip if short on time)*.

In [ ]:
ks = range(2, 7)
scores = [silhouette_score(X_std, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X_std))
          for k in ks]

plt.plot(list(ks), scores, "o-", color="#2a7f62", linewidth=2)
plt.title("Silhouette score by number of clusters (higher is better)")
plt.xlabel("k"); plt.ylabel("Silhouette score")
plt.show()

## 5. Do the two labellers agree?

Two methods that know nothing about each other. Where they **agree**, we can trust the label more than either one alone.

In [ ]:
agree = df["threshold_label"] == df["cluster_label"]
ari = adjusted_rand_score(df["threshold_label"], df["cluster_label"])
print(f"Agreement:           {agree.mean():.1%} of clips ({agree.sum()} / {len(df)})")
print(f"Adjusted Rand Index: {ari:.3f}")

order = ["Low", "High"]
cm = confusion_matrix(df["threshold_label"], df["cluster_label"], labels=order)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order, annot_kws={"size": 18})
plt.xlabel("Clustering says"); plt.ylabel("Threshold says")
plt.title("Where the two labellers agree")
plt.show()

## 6. Settle the disagreements

1. Train a **Random Forest** only on the clips where both labellers agree
2. Check it on held-out agreed clips
3. Let it decide the clips where they disagreed

In [ ]:
consensus = df[agree]
disputed  = df[~agree].copy()

X_tr, X_val, y_tr, y_val = train_test_split(
    consensus[FEATURES], consensus["threshold_label"],
    test_size=0.2, random_state=SEED, stratify=consensus["threshold_label"],
)
clf = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_tr, y_tr)
print(f"Random Forest accuracy on held-out agreed clips: {clf.score(X_val, y_val):.1%}")

# Assign predictions as a NumPy array so each one lands on the right row
disputed["final_label"] = clf.predict(disputed[FEATURES])

df["final_label"] = df["threshold_label"]
df.loc[disputed.index, "final_label"] = disputed["final_label"]

print(f"\n{len(disputed)} disputed clips resolved:")
print(disputed["final_label"].value_counts().to_string())
print("\nFinal training labels:")
print(df["final_label"].value_counts().to_string())

In [ ]:
# A peek at a few clips the labellers argued about
cols = ["filename", "activity_score", "threshold_label", "cluster_label", "final_label"]
df.loc[disputed.index, cols].head(8).round(3)

### The recipe, for any unlabelled dataset
1. **Domain features:** what do experts already measure?
2. **Two independent labellers:** a rule and a clustering method
3. **Measure agreement:** trust the consensus
4. **Learn from the consensus** to settle the rest

These final labels are what the image model (InceptionV3 on mel spectrograms) was trained on.

## 7. *(Optional)* Ask the deployed model

The trained model is served with **FastAPI** in a Docker container on Hugging Face Spaces.

In [ ]:
import requests

try:
    with open(HIGH_CLIP, "rb") as f:
        r = requests.post(f"{API_URL}/predict", files={"file": (HIGH_CLIP.name, f, "audio/wav")}, timeout=60)
    r.raise_for_status()
    print(r.json())
except requests.RequestException as e:
    print(f"API not reachable right now ({e.__class__.__name__}). Show the slide instead.")